# Aula 18 — Hyperparameter tuning: Grid Search, Random Search e validação aninhada

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/18-hyperparameter-tuning-laboratorio.ipynb)

**Pergunta:** como selecionar hiperparâmetros sem chamar o melhor score interno de desempenho final?

Hipóteses pré-registradas:

1. o máximo entre muitas estimativas ruidosas ficará otimista, mesmo quando todos os candidatos forem equivalentes;
2. Grid e Random Search serão comparadas com 16 candidatos, os mesmos quatro folds e ROC-AUC primária;
3. nested CV separará a escolha interna da estimativa externa;
4. o teste externo será consultado uma vez, depois de congelar a estratégia Random Search.


## Ambiente e dependências

- Python ≥ 3.10
- NumPy ≥ 1.24
- SciPy ≥ 1.10
- Matplotlib ≥ 3.7
- scikit-learn ≥ 1.3

Os dados são sintéticos, locais e reproduzíveis. Para evitar paralelismo aninhado e tornar o orçamento auditável, todas as buscas usam n_jobs igual a 1.


In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scipy
import sklearn
from scipy.stats import loguniform
from sklearn.datasets import make_classification
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("error")
SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 1. Winner's curse em uma simulação controlada

Todos os candidatos têm score verdadeiro 0,75. A validação observa esse valor com ruído normal de desvio 0,03. Em cada repetição escolhemos o maior score e medimos novamente o candidato selecionado em uma amostra independente.


In [ ]:
true_score = 0.75
noise_sd = 0.03
n_repetitions = 5000
candidate_counts = np.array([1, 10, 100, 1000])
selected_validation_means = []
independent_means = []

for count in candidate_counts:
    validation = true_score + rng.normal(
        0, noise_sd, size=(n_repetitions, count)
    )
    selected_validation_means.append(validation.max(axis=1).mean())
    independent = true_score + rng.normal(0, noise_sd, size=n_repetitions)
    independent_means.append(independent.mean())

selected_validation_means = np.array(selected_validation_means)
independent_means = np.array(independent_means)
for count, val, ind in zip(
    candidate_counts, selected_validation_means, independent_means
):
    print(f"{count:4d} candidatos | selecionado={val:.6f} | independente={ind:.6f}")

assert selected_validation_means[-1] > true_score + 0.08
assert abs(independent_means[-1] - true_score) < 0.003

### Visualização

O score selecionado cresce com o número de tentativas; a remedição independente continua próxima do valor verdadeiro. Mais candidatos não criaram modelos melhores nesta simulação — apenas mais oportunidades para ruído favorável.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(candidate_counts, selected_validation_means, "o-", label="máximo na validação")
ax.plot(candidate_counts, independent_means, "o-", label="remedição independente")
ax.axhline(true_score, color="black", linestyle="--", label="score verdadeiro")
ax.set_xscale("log")
ax.set(
    xlabel="Número de candidatos (escala log)",
    ylabel="Score médio",
    title="Seleção também pode overfitar a validação",
)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Dataset e teste externo lacrado

Geramos uma classificação binária difícil, com 60 features, ruído de rótulo e apenas cinco features informativas. O teste de 20% é separado antes das buscas. A estratégia final já está fixada: Random Search com distribuição log-uniforme.


In [ ]:
X, y = make_classification(
    n_samples=450,
    n_features=60,
    n_informative=5,
    n_redundant=5,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[0.65, 0.35],
    class_sep=0.65,
    flip_y=0.12,
    random_state=SEED,
)
indices = np.arange(len(y))
idx_dev, idx_test = train_test_split(
    indices, test_size=0.20, stratify=y, random_state=SEED + 1
)
X_dev, y_dev = X[idx_dev], y[idx_dev]
X_test, y_test = X[idx_test], y[idx_test]

print("Desenvolvimento:", X_dev.shape, "prevalência:", f"{y_dev.mean():.6f}")
print("Teste lacrado:  ", X_test.shape, "prevalência:", f"{y_test.mean():.6f}")
assert set(idx_dev).isdisjoint(idx_test)
assert len(idx_dev) + len(idx_test) == len(y)

## 3. Pipeline e espaços

O scaler pertence ao pipeline e será ajustado novamente em cada fold. A grade tem 4 × 4 = 16 candidatos. Random Search também terá 16 candidatos, amostrando C e gamma em escala logarítmica.


In [ ]:
pipeline = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", class_weight="balanced", cache_size=500),
)
grid_space = {
    "svc__C": [0.03, 0.3, 3.0, 30.0],
    "svc__gamma": [0.001, 0.01, 0.1, 1.0],
}
random_space = {
    "svc__C": loguniform(1e-4, 1e3),
    "svc__gamma": loguniform(1e-5, 1e1),
}
n_candidates = 16
n_inner_folds = 4
fits_per_search = n_candidates * n_inner_folds + 1

print("Candidatos por método:", n_candidates)
print("Fits por busca, incluindo refit:", fits_per_search)
assert len(grid_space["svc__C"]) * len(grid_space["svc__gamma"]) == 16
assert fits_per_search == 65

### 3.1 O que log-uniforme significa

Sorteamos 20 mil valores apenas para verificar a distribuição. Aproximadamente metade deve cair abaixo da média geométrica entre os limites; a mediana é multiplicativa, não aritmética.


In [ ]:
sample_rng = np.random.default_rng(SEED + 2)
low, high = 1e-4, 1e3
sampled_c = loguniform(low, high).rvs(size=20_000, random_state=sample_rng)
geometric_midpoint = np.sqrt(low * high)
fraction_below = np.mean(sampled_c < geometric_midpoint)

print(f"Limites: [{sampled_c.min():.8f}, {sampled_c.max():.3f}]")
print(f"Média geométrica: {geometric_midpoint:.6f}")
print(f"Fração abaixo da média geométrica: {fraction_below:.6f}")
assert sampled_c.min() >= low and sampled_c.max() <= high
assert 0.48 < fraction_below < 0.52

## 4. Grid e Random com o mesmo orçamento

As duas buscas usam o mesmo pipeline, os mesmos quatro folds e duas métricas. ROC-AUC é a métrica primária e controla o refit; F1 é apenas diagnóstico secundário.


In [ ]:
inner_cv = StratifiedKFold(
    n_splits=n_inner_folds, shuffle=True, random_state=SEED + 3
)
scoring = {"auc": "roc_auc", "f1": "f1"}

grid_search = GridSearchCV(
    pipeline,
    grid_space,
    cv=inner_cv,
    scoring=scoring,
    refit="auc",
    n_jobs=1,
    error_score="raise",
    return_train_score=True,
)
random_search = RandomizedSearchCV(
    pipeline,
    random_space,
    n_iter=n_candidates,
    cv=inner_cv,
    scoring=scoring,
    refit="auc",
    random_state=SEED + 4,
    n_jobs=1,
    error_score="raise",
    return_train_score=True,
)

grid_search.fit(X_dev, y_dev)
random_search.fit(X_dev, y_dev)

print("Grid  | best AUC:", f"{grid_search.best_score_:.6f}",
      "|", grid_search.best_params_)
print("Random| best AUC:", f"{random_search.best_score_:.6f}",
      "|", random_search.best_params_)
assert len(grid_search.cv_results_["params"]) == n_candidates
assert len(random_search.cv_results_["params"]) == n_candidates

### 4.1 Auditoria de cv_results_

O ranking sozinho não basta. Exibimos média de AUC, desvio entre folds, F1 e diferença treino–validação dos três primeiros colocados de cada método.


In [ ]:
def top_rows(search, top_n=3):
    results = search.cv_results_
    order = np.argsort(results["rank_test_auc"])
    rows = []
    for i in order[:top_n]:
        rows.append({
            "rank": int(results["rank_test_auc"][i]),
            "auc": float(results["mean_test_auc"][i]),
            "auc_std": float(results["std_test_auc"][i]),
            "f1": float(results["mean_test_f1"][i]),
            "gap_train_val": float(
                results["mean_train_auc"][i] - results["mean_test_auc"][i]
            ),
            "params": results["params"][i],
        })
    return rows

for name, search in [("Grid", grid_search), ("Random", random_search)]:
    print(f"\n{name}")
    for row in top_rows(search):
        print(
            f"rank={row['rank']} auc={row['auc']:.6f} "
            f"std={row['auc_std']:.6f} f1={row['f1']:.6f} "
            f"gap={row['gap_train_val']:.6f} params={row['params']}"
        )

random_pairs = {
    (float(c), float(g))
    for c, g in zip(
        random_search.cv_results_["param_svc__C"],
        random_search.cv_results_["param_svc__gamma"],
    )
}
assert len(random_pairs) == n_candidates

### 4.2 Superfície discreta da grade

O mapa mostra que extremos de regularização podem falhar. Ele visualiza somente os 16 pontos da grade; não representa todo o espaço contínuo.


In [ ]:
grid_auc = np.asarray(grid_search.cv_results_["mean_test_auc"]).reshape(4, 4)
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(grid_auc, cmap="viridis", aspect="auto")
ax.set(
    xticks=np.arange(4),
    xticklabels=grid_space["svc__gamma"],
    yticks=np.arange(4),
    yticklabels=grid_space["svc__C"],
    xlabel="gamma",
    ylabel="C",
    title="ROC-AUC média nos 16 pontos da grade",
)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{grid_auc[i, j]:.3f}", ha="center", va="center",
                color="white" if grid_auc[i, j] < grid_auc.mean() else "black")
fig.colorbar(image, ax=ax, label="ROC-AUC")
plt.tight_layout()
plt.show()

assert grid_auc.shape == (4, 4)

## 5. Nested cross-validation manual

Em cada um dos cinco folds externos, uma nova Random Search com 24 candidatos e três folds internos seleciona hiperparâmetros usando apenas o treino externo. O candidato refitado prevê o fold externo intocado.


In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 5)
n_nested_candidates = 24
n_nested_inner = 3
nested_fit_budget = 5 * (n_nested_candidates * n_nested_inner + 1)
inner_best_scores = []
outer_scores = []
chosen_params = []
outer_overlap = []

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_dev, y_dev), start=1):
    fold_inner = StratifiedKFold(
        n_splits=n_nested_inner,
        shuffle=True,
        random_state=SEED + 100 + fold,
    )
    fold_search = RandomizedSearchCV(
        pipeline,
        random_space,
        n_iter=n_nested_candidates,
        cv=fold_inner,
        scoring="roc_auc",
        refit=True,
        random_state=SEED + 200 + fold,
        n_jobs=1,
        error_score="raise",
    )
    fold_search.fit(X_dev[train_idx], y_dev[train_idx])
    outer_score = roc_auc_score(
        y_dev[val_idx], fold_search.decision_function(X_dev[val_idx])
    )
    inner_best_scores.append(fold_search.best_score_)
    outer_scores.append(outer_score)
    chosen_params.append(fold_search.best_params_)
    outer_overlap.append(len(np.intersect1d(train_idx, val_idx)))

inner_best_scores = np.array(inner_best_scores)
outer_scores = np.array(outer_scores)
optimism_gap = inner_best_scores.mean() - outer_scores.mean()

for fold, (inner_score, outer_score, params) in enumerate(
    zip(inner_best_scores, outer_scores, chosen_params), start=1
):
    print(
        f"Fold {fold}: interno={inner_score:.6f} "
        f"externo={outer_score:.6f} params={params}"
    )
print("Orçamento nested:", nested_fit_budget, "fits")
print(f"Interno médio: {inner_best_scores.mean():.6f}")
print(f"Externo médio: {outer_scores.mean():.6f}")
print(f"Gap interno - externo: {optimism_gap:.6f}")

assert nested_fit_budget == 365
assert np.all(np.array(outer_overlap) == 0)
assert optimism_gap > 0

### 5.1 O que varia entre folds

Os candidatos escolhidos podem mudar porque cada loop interno recebe dados diferentes. A distribuição externa estima o procedimento de busca; os scores internos diagnosticam a seleção.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
folds = np.arange(1, 6)
ax.plot(folds, inner_best_scores, "o-", label="melhor score interno")
ax.plot(folds, outer_scores, "o-", label="score externo")
ax.axhline(outer_scores.mean(), color="black", linestyle="--",
           label="média externa")
ax.set(
    xticks=folds,
    xlabel="Fold externo",
    ylabel="ROC-AUC",
    title="Seleção interna versus estimativa externa",
)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

unique_choices = {
    (round(float(p["svc__C"]), 8), round(float(p["svc__gamma"]), 8))
    for p in chosen_params
}
print("Configurações distintas escolhidas:", len(unique_choices))
print(f"ROC-AUC externa: {outer_scores.mean():.6f} ± {outer_scores.std(ddof=1):.6f}")

## 6. Busca final e teste externo único

O protocolo Random Search, seu espaço, orçamento, folds e métrica foram congelados antes do teste. Agora a busca é refeita em todo o desenvolvimento e seu best_estimator_ é avaliado uma vez.


In [ ]:
final_search = RandomizedSearchCV(
    pipeline,
    random_space,
    n_iter=n_candidates,
    cv=inner_cv,
    scoring="roc_auc",
    refit=True,
    random_state=SEED + 4,
    n_jobs=1,
    error_score="raise",
)
final_search.fit(X_dev, y_dev)
test_scores = final_search.decision_function(X_test)
test_predictions = final_search.predict(X_test)
test_auc = roc_auc_score(y_test, test_scores)
test_f1 = f1_score(y_test, test_predictions)

print("Parâmetros finais:", final_search.best_params_)
print(f"Melhor AUC interna: {final_search.best_score_:.6f}")
print(f"AUC nested externa: {outer_scores.mean():.6f} ± {outer_scores.std(ddof=1):.6f}")
print(f"AUC no teste:        {test_auc:.6f}")
print(f"F1 no teste:         {test_f1:.6f}")
assert final_search.best_params_ == random_search.best_params_
assert 0.5 < test_auc <= 1.0

## 7. Verificações consolidadas

Os asserts abaixo verificam orçamento, independência do teste, unicidade das amostras aleatórias, ausência de sobreposição externa e o comportamento da simulação.


In [ ]:
checks = {
    "seed fixa": SEED == 20260908,
    "teste disjunto": set(idx_dev).isdisjoint(idx_test),
    "orçamento Grid": len(grid_search.cv_results_["params"]) == 16,
    "orçamento Random": len(random_search.cv_results_["params"]) == 16,
    "amostras Random únicas": len(random_pairs) == 16,
    "folds externos disjuntos": all(value == 0 for value in outer_overlap),
    "winner's curse simulada": selected_validation_means[-1] > true_score + 0.08,
    "nested executada": len(outer_scores) == 5,
}
for name, passed in checks.items():
    print(f"{name:27s}: {'OK' if passed else 'FALHOU'}")
assert all(checks.values())

## Conclusões sustentadas

- Selecionar o máximo entre estimativas ruidosas gera otimismo, mesmo sem diferença real entre candidatos.
- Grid e Random podem ser comparadas de modo justo somente com pipeline, folds, métrica e orçamento comuns.
- O melhor score interno mede a evidência que escolheu a configuração.
- Nested CV mede a busca inteira em dados externos ao loop de seleção.
- O teste preserva sua função apenas quando consultado depois do congelamento das decisões.

**Não sustentado:** um único dataset sintético não prova superioridade universal de Grid ou Random Search. A configuração final também não é uma constante científica; ela depende dos dados e do protocolo.


## Desafio

1. mantenha 16 candidatos e troque as distribuições;
2. repita Random Search com cinco seeds sem usar o teste para escolher uma;
3. altere o número de folds internos e recalcule o orçamento;
4. substitua SVC por árvore e construa um espaço com inteiros e categorias;
5. imponha uma política pré-registrada que maximize AUC entre candidatos com F1 mínimo.

Registre cv_results_, índices, tempos e conclusão. Nunca ajuste o espaço após consultar o teste.


## Referências

- [scikit-learn — Tuning hyper-parameters](https://scikit-learn.org/stable/modules/grid_search.html)
- [scikit-learn — Nested versus non-nested CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [SciPy — loguniform](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.loguniform.html)
- [Bergstra e Bengio (2012), JMLR](https://jmlr.org/papers/v13/bergstra12a.html)
- [Cawley e Talbot (2010), JMLR](https://jmlr.org/papers/v11/cawley10a.html)
